In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler  
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

In [4]:

redwine_df = pd.read_csv("winequality-red.csv")
redwine_df =redwine_df.drop_duplicates().dropna()
redwine_df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5


In [5]:
X = redwine_df.drop("quality", axis=1)
y = redwine_df["quality"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


X_train: (1087, 11)
X_test : (272, 11)
y_train: (1087,)
y_test : (272,)


In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
kernels = ["linear", "poly", "rbf", "sigmoid"]
c_values = [0.1, 1, 10]

results = []

for kernel in kernels:
    for i, c in enumerate(c_values, start=1):
        c_label = f"c{i}"
        model = SVR(kernel=kernel, C=c)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        
        mse = mean_squared_error(y_test, y_pred)

        results.append({
            "Type of Kernel": kernel.capitalize(),
            "c_label": c_label,
            "C": c,
            "MSE_Result" : round(mse, 4)
        })

results_df = pd.DataFrame(results)
results_df
model.fit

<bound method BaseLibSVM.fit of SVR(C=10, kernel='sigmoid')>

In [8]:
results_table = results_df.pivot(index="Type of Kernel", columns="c_label", values="MSE_Result")
results_table = results_table.reindex(index=["Linear", "Poly", "Rbf", "Sigmoid"])
for i, c in enumerate(c_values):
    print(f"c{i+1} = {c}")
results_table

c1 = 0.1
c2 = 1
c3 = 10


c_label,c1,c2,c3
Type of Kernel,,,
Linear,0.4243,0.4240,0.4240
Poly,0.4836,0.4772,0.5836
Rbf,0.4322,0.4173,0.4624
Sigmoid,0.9812,90.1252,7726.9407


In [9]:
best_row = results_df.loc[results_df["MSE_Result"].idxmin()]
best_kernel = best_row["Type of Kernel"].lower()
best_c = best_row["C"]


print(f"Best SVM: kernel = {best_kernel} \nC = {best_c}\nMSE = {best_row['MSE_Result']}")

Best SVM: kernel = rbf 
C = 1.0
MSE = 0.4173
